# Session 1 — Data Exploration

**Goal:** Understand the Telco Customer Churn dataset before building any models.

Key questions to answer:
- What does the target variable look like? (Class balance)
- Which features might be strong predictors of churn?
- Are there any data quality issues we need to handle?

In [0]:
%run ../utils/config

## Understand the shape of the data
Look at the number of rows and columns as well as a sampling of the data in the table

In [0]:
df = spark.table(f"00_shared.telco_churn")

print(f"Shape: {df.count():,} rows × {len(df.columns)} columns")
display(df.limit(10))

## Target Variable: Churn

First, let's check the class balance. A significant imbalance will affect how we choose and evaluate our models.

In [0]:
from pyspark.sql import functions as F

churn_counts = df.groupBy("Churn").count().orderBy("Churn")
display(churn_counts)

# Calculate churn rate
total = df.count()
churned = df.filter(F.col("Churn") == "Yes").count()
print(f"\nChurn rate: {churned/total:.1%}  ({churned:,} / {total:,} customers)")
print("\nℹ️  ~26% churn rate — moderately imbalanced.")
print("   This means accuracy alone is misleading — we'll focus on F1 and Recall.")

## Contract Type vs Churn

Intuitively, customers on month-to-month contracts should churn more easily than those locked into annual contracts.

In [0]:
contract_churn = (
    df.groupBy("Contract", "Churn")
      .count()
      .orderBy("Contract", "Churn")
)
display(contract_churn)

Databricks visualization. Run in Databricks to view.

## Numeric Feature Distributions

`tenure`, `MonthlyCharges`, and `TotalCharges` are our numeric features.
Let's compare their distributions for churned vs non-churned customers.

In [0]:
numeric_stats = df.groupBy("Churn").agg(
    F.round(F.mean("tenure"), 1).alias("avg_tenure"),
    F.round(F.mean("MonthlyCharges"), 2).alias("avg_monthly_charges"),
    F.round(F.mean("TotalCharges"), 2).alias("avg_total_charges"),
    F.count("*").alias("count")
).orderBy("Churn")

# Bar chart: compare avg_tenure, avg_monthly_charges, avg_total_charges by Churn
display(numeric_stats)

print("\nKey insight: churned customers have much shorter tenure on average.")
print("Short tenure + high monthly charges = high churn risk signal.")

## Data Quality: Null Values

`TotalCharges` can be null for brand-new customers (tenure = 0).
Our feature pipeline handles this with median imputation — but let's verify.

In [0]:
null_counts = df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).toPandas().T.rename(columns={0: "null_count"})
null_counts = null_counts[null_counts["null_count"] > 0]

if len(null_counts) == 0:
    print("✓ No null values found.")
else:
    print("Columns with nulls:")
    print(null_counts)
    print("\n→ TotalCharges nulls = new customers. Our pipeline imputes with the median.")

## Key Takeaways

Before we build any model, we know:

1. **Class imbalance**: ~26% churn. Use F1/Recall, not accuracy.
2. **Strong predictors**: Contract type, tenure, MonthlyCharges
3. **Data quality**: TotalCharges has some nulls → median imputation needed
4. **Business context**: Month-to-month customers churn at ~3× the rate of annual customers

➡️ Next: [02_messy_notebook]($./02_messy_notebook) (instructor demo) → `03_refactored_pipeline.ipynb`